# ScoutLite — Runnable Demo

**A player research brief for a scout to weigh — not a scouting verdict.**

This notebook is a self-contained walkthrough of the ScoutLite pipeline (PE6201 course project): player name → FBref stats/bio → Understat xG/xA → a non-AI percentile-based Quality signal → NewsAPI recent headlines → one LLM synthesis call (news read + Fit signal) → a downloadable Word doc research brief.

It writes each project module to disk with `%%writefile` so the whole thing runs from this one notebook — no repo checkout needed. The module code itself is unchanged from the local project; only the setup (Chrome install, API keys) is Colab-specific.

**⚠️ One honest caveat before you run this.** The FBref scraping step relies on `seleniumbase`'s undetected-Chrome mode getting past Cloudflare's bot challenge. That was verified reliably from a local machine's residential-ish connection — it has **not** been verified from Colab's cloud IP ranges, which Cloudflare's bot management sometimes treats with more suspicion. It may just work. If the FBref cell hangs or returns a Cloudflare challenge page instead of real data, that's this exact risk, not a bug elsewhere in the pipeline — everything downstream (Understat, scoring, NewsAPI, the LLM call, the Word doc) is independent of it and was verified separately.

**Access-decision context** (full reasoning in the project's `NOTES.md`): FBref scraping is permitted but rate-limited (paced accordingly below); Understat's `robots.txt` blanket-disallows scraping and this pipeline fetches it anyway as a deliberate, disclosed override; Transfermarkt/Reddit/X/FotMob were evaluated and are not used.

## 1. Setup

Installs a Chrome binary (for `seleniumbase`'s undetected-Chrome driver to control) and every Python package the pipeline needs.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq chromium-browser
!pip install -q seleniumbase pandas lxml openai python-dotenv requests vaderSentiment streamlit python-docx soccerdata beautifulsoup4

## 2. Write the project modules to disk

Each cell below is the exact, unmodified source of one file from the local ScoutLite project. `%%writefile` saves it to the Colab runtime's disk so the normal `import` statements a few cells down work as-is.

In [ ]:
%%writefile scoutlite.py
#!/usr/bin/env python3
"""
ScoutLite MVP slice: one player name -> FBref season stats -> one LLM call -> one paragraph.

Data source: FBref only. Scraping is permitted but rate-limited to <10 requests/min
(violations risk a block of up to 24h), so every request is paced at ~6.5-8s with jitter.
FBref sits behind Cloudflare's bot challenge, which blocks plain HTTP (requests/curl) and
standard headless automation (Playwright/Selenium) alike -- only an undetected browser
driver (seleniumbase's uc=True mode) gets through, so that's what this script uses.
"""
import argparse
import os
import random
import re
import sys
import time
from pathlib import Path
from urllib.parse import quote

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from seleniumbase import Driver

RATE_LIMIT_SECONDS = 6.5
JITTER_SECONDS = 1.5

ROOT = Path(__file__).resolve().parent
load_dotenv(ROOT / ".env")


def pace():
    delay = RATE_LIMIT_SECONDS + random.uniform(0, JITTER_SECONDS)
    print(f"[pacing] sleeping {delay:.1f}s before FBref request (rate-limit compliance)...")
    time.sleep(delay)


def fetch_player_page_html(player_name: str) -> tuple[str, str]:
    """Return (final_url, html) for a player's FBref page via the site search."""
    search_url = f"https://fbref.com/en/search/search.fcgi?search={quote(player_name)}"
    driver = Driver(uc=True, headless=True)
    try:
        pace()
        driver.uc_open_with_reconnect(search_url, reconnect_time=6)
        driver.sleep(3)
        current_url = driver.get_current_url()

        if "/search/" in current_url:
            # Ambiguous name: multiple matches, take the first result.
            html = driver.get_page_source()
            soup = BeautifulSoup(html, "lxml")
            link = soup.select_one("div.search-item-name a")
            if not link:
                raise RuntimeError(f"No FBref match found for '{player_name}'")
            player_url = "https://fbref.com" + link["href"]
            pace()
            driver.uc_open_with_reconnect(player_url, reconnect_time=6)
            driver.sleep(3)
            current_url = driver.get_current_url()

        return current_url, driver.get_page_source()
    finally:
        driver.quit()


def _season_rows(soup: BeautifulSoup, table_id: str):
    """Return every real-season tbody row (in table order, oldest first) for the given FBref
    table, or [] if the table isn't on this page (e.g. Goalkeeping, for an outfield player)."""
    table = soup.find("table", id=table_id)
    if table is None:
        return []
    return [
        tr
        for tr in table.find("tbody").find_all("tr")
        if (cell := tr.find(attrs={"data-stat": "year_id"}))
        and re.match(r"^\d{4}(-\d{4})?$", cell.text.strip())
    ]


def _season_row(soup: BeautifulSoup, table_id: str, season: str | None = None):
    """Return the row for a specific season (year_id), or the most recent one if season is
    None. None if the table isn't on the page or the requested season isn't in it."""
    rows = _season_rows(soup, table_id)
    if not rows:
        return None
    if season is None:
        return rows[-1]
    return next((r for r in rows if r.find(attrs={"data-stat": "year_id"}).text.strip() == season), None)


def list_available_seasons(html: str) -> list[str]:
    """All seasons present in the Standard Stats table, most recent first."""
    soup = BeautifulSoup(html, "lxml")
    rows = _season_rows(soup, "stats_standard_dom_lg")
    return [r.find(attrs={"data-stat": "year_id"}).text.strip() for r in reversed(rows)]


def _row_stats(row, fields: dict) -> dict:
    """fields maps output key -> FBref data-stat name."""
    def stat(name):
        cell = row.find(attrs={"data-stat": name})
        return cell.text.strip() if cell else ""

    return {key: stat(data_stat) for key, data_stat in fields.items()}


def extract_latest_season(html: str, season: str | None = None) -> dict:
    """Pull a season row (most recent if season is None) from the Standard Stats table."""
    soup = BeautifulSoup(html, "lxml")
    row = _season_row(soup, "stats_standard_dom_lg", season)
    if row is None:
        raise RuntimeError(
            "Could not find season rows in the standard stats table"
            if season is None
            else f"Season '{season}' not found in the standard stats table"
        )

    return _row_stats(row, {
        "season": "year_id",
        "age": "age",
        "squad": "team",
        "competition": "comp_level",
        "matches_played": "games",
        "starts": "games_starts",
        "minutes": "minutes",
        "goals": "goals",
        "assists": "assists",
        "goals_plus_assists": "goals_assists",
        "non_penalty_goals": "goals_pens",
        "yellow_cards": "cards_yellow",
        "red_cards": "cards_red",
    })


def extract_misc_stats(html: str, season: str | None = None) -> dict | None:
    """Pull defensive/discipline numbers (tackles, interceptions, fouls, etc.) from the Misc
    table -- applies to any position, most relevant for outfield defensive contribution."""
    soup = BeautifulSoup(html, "lxml")
    row = _season_row(soup, "stats_misc_dom_lg", season)
    if row is None:
        return None

    return _row_stats(row, {
        "fouls_committed": "fouls",
        "fouls_drawn": "fouled",
        "offsides": "offsides",
        "crosses": "crosses",
        "interceptions": "interceptions",
        "tackles_won": "tackles_won",
        "penalties_won": "pens_won",
        "penalties_conceded": "pens_conceded",
        "own_goals": "own_goals",
    })


def extract_keeper_stats(html: str, season: str | None = None) -> dict | None:
    """Pull goalkeeping numbers from the Goalkeeping table -- only present on the page for
    players who are actually goalkeepers."""
    soup = BeautifulSoup(html, "lxml")
    row = _season_row(soup, "stats_keeper_dom_lg", season)
    if row is None:
        return None

    return _row_stats(row, {
        "gk_matches_played": "gk_games",
        "gk_starts": "gk_games_starts",
        "gk_minutes": "gk_minutes",
        "goals_against": "gk_goals_against",
        "shots_on_target_against": "gk_shots_on_target_against",
        "saves": "gk_saves",
        "save_pct": "gk_save_pct",
        "wins": "gk_wins",
        "draws": "gk_ties",
        "losses": "gk_losses",
        "clean_sheets": "gk_clean_sheets",
        "clean_sheet_pct": "gk_clean_sheets_pct",
        "penalties_faced": "gk_pens_att",
        "penalties_allowed": "gk_pens_allowed",
        "penalties_saved": "gk_pens_saved",
    })


def extract_player_bio(html: str) -> dict:
    """Pull background info (name, physical profile, birth, nationality, club, contract) from
    the #meta block at the top of an FBref player page."""
    soup = BeautifulSoup(html, "lxml")
    meta = soup.find("div", id="meta")
    if meta is None:
        raise RuntimeError("Could not find the player bio block on the FBref page")

    text = meta.get_text("\n", strip=True)

    # The full-name <p> is the one whose entire content is just a <strong> tag, with no
    # "Label:" prefix (unlike Position/Footed/Club/etc which all follow a labeled <p>).
    full_name = ""
    for p in meta.find_all("p"):
        strong = p.find("strong")
        if strong and strong.get_text(strip=True) == p.get_text(strip=True):
            full_name = strong.get_text(strip=True)
            break

    position_match = re.search(r"Position:\s*([^\u25aa\n]+)", text)
    foot_match = re.search(r"Footed:\s*(\w+)", text)

    height_weight = ""
    hw_p = next((p for p in meta.find_all("p") if re.search(r"\d+cm", p.get_text())), None)
    if hw_p:
        hw_match = re.search(r"(\d+cm)\s*,?\s*(\d+kg)", hw_p.get_text(" ", strip=True))
        if hw_match:
            height_weight = f"{hw_match.group(1)}, {hw_match.group(2)}"

    birth_span = meta.find(id="necro-birth")
    birth_date = birth_span["data-birth"] if birth_span and birth_span.has_attr("data-birth") else ""

    birthplace = ""
    if birth_span:
        born_p = birth_span.find_parent("p")
        if born_p:
            place_span = next(
                (s for s in born_p.find_all("span") if s.get_text(strip=True).startswith("in ")),
                None,
            )
            if place_span:
                birthplace = place_span.get_text(strip=True).removeprefix("in ").strip()

    club_match = re.search(r"Club:\s*([^\n]+)", text)
    nat_team_match = re.search(r"National Team:\s*([^\n]+)", text)
    contract_match = re.search(r"Expires\s+([A-Za-z]+\s+\d{4})", text)

    return {
        "full_name": full_name,
        "position": position_match.group(1).strip() if position_match else "",
        "footed": foot_match.group(1).strip() if foot_match else "",
        "height_weight": height_weight,
        "birth_date": birth_date,
        "birthplace": birthplace,
        "national_team": nat_team_match.group(1).strip() if nat_team_match else "",
        "club": club_match.group(1).strip() if club_match else "",
        "contract_expires": contract_match.group(1) if contract_match else "",
    }


def summarize_with_llm(player_name: str, stats: dict) -> str:
    client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")
    stats_lines = "\n".join(f"- {k.replace('_', ' ')}: {v}" for k, v in stats.items())
    prompt = (
        f"You are a football scouting assistant. Using ONLY the stats listed below, write a "
        f"short paragraph (3-5 sentences) summarizing {player_name}'s most recent season. Cite "
        f"specific numbers from the stats. Do not invent or infer any stat that is not listed. "
        f"Do not speculate about transfer value, potential, or future performance.\n\n"
        f"Player: {player_name}\nStats:\n{stats_lines}"
    )
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    return response.choices[0].message.content.strip()


def main():
    parser = argparse.ArgumentParser(
        description="ScoutLite MVP slice: player name -> FBref stats -> LLM paragraph"
    )
    parser.add_argument("player", help="Player name, e.g. 'Erling Haaland'")
    args = parser.parse_args()

    if not os.environ.get("DEEPSEEK_API_KEY"):
        sys.exit("DEEPSEEK_API_KEY is not set. Add it to .env in this project folder.")

    print(f"Fetching FBref data for '{args.player}'...")
    url, html = fetch_player_page_html(args.player)
    print(f"Resolved to: {url}")

    stats = extract_latest_season(html)
    print(f"Latest season found: {stats['season']} ({stats['squad']}, {stats['competition']})")

    print("Calling DeepSeek-V3 for the summary paragraph...")
    paragraph = summarize_with_llm(args.player, stats)

    output_dir = ROOT / "output"
    output_dir.mkdir(exist_ok=True)
    slug = re.sub(r"[^a-z0-9]+", "_", args.player.lower()).strip("_")
    out_path = output_dir / f"{slug}.txt"
    out_path.write_text(paragraph + "\n")

    print("\n--- Summary ---")
    print(paragraph)
    print(f"\nSaved to {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile understat_xg.py
#!/usr/bin/env python3
"""
Understat xG/xA lookup, by player name within a league+season.

IMPORTANT -- access note: understat.com's robots.txt is a blanket `Disallow: /` for every
user agent, no exceptions. This module fetches it anyway, on the project owner's explicit
direction after being shown that finding -- it is a deliberate override, not an oversight.
See NOTES.md for the full reasoning.

Mechanically this is simple: understat.com's league pages load their data from a JSON
endpoint (getLeagueData/<league>/<year>) rather than a static page. It needs a session
cookie from the league page first and a Referer header, but no browser automation, no
Cloudflare, no rate-limit wall was observed.

Coverage: only the 6 leagues Understat tracks (top 5 European leagues + Russian Premier
League). Players outside those leagues will not be found here.
"""
import unicodedata

import requests

UNDERSTAT_LEAGUES = {
    "premier league": "EPL",
    "la liga": "La liga",
    "bundesliga": "Bundesliga",
    "serie a": "Serie A",
    "ligue 1": "Ligue 1",
    "russian premier league": "RFPL",
}


def fbref_comp_to_understat_league(comp_level: str) -> str | None:
    """Map an FBref 'comp_level' string (e.g. '1. Premier League') to an Understat league slug."""
    name = comp_level.split(".", 1)[-1].strip().lower()
    return UNDERSTAT_LEAGUES.get(name)


def fbref_season_to_understat_year(season: str) -> str:
    """Map an FBref season string ('2025-2026' or '2026') to Understat's year param (start year)."""
    return season.split("-")[0]


def _normalize_name(name: str) -> str:
    decomposed = unicodedata.normalize("NFKD", name)
    stripped = "".join(c for c in decomposed if not unicodedata.combining(c))
    return stripped.lower().strip()


def fetch_league_players(league: str, year: str) -> list[dict]:
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"})
    league_url = f"https://understat.com/league/{league}"
    session.get(league_url, timeout=15)  # picks up session cookies

    api_url = f"https://understat.com/getLeagueData/{league}/{year}"
    response = session.get(
        api_url,
        headers={"Referer": league_url, "X-Requested-With": "XMLHttpRequest"},
        timeout=15,
    )
    response.raise_for_status()
    return response.json().get("players", [])


def find_player_xg(players: list[dict], player_name: str, team_hint: str = "") -> dict | None:
    target = _normalize_name(player_name)
    team_target = _normalize_name(team_hint) if team_hint else ""

    candidates = [p for p in players if _normalize_name(p["player_name"]) == target]
    if not candidates:
        return None
    if len(candidates) > 1 and team_target:
        narrowed = [p for p in candidates if team_target in _normalize_name(p["team_title"])]
        if narrowed:
            candidates = narrowed

    p = candidates[0]
    return {
        "understat_team": p["team_title"],
        "games": p["games"],
        "minutes": p["time"],
        "goals": p["goals"],
        "xG": round(float(p["xG"]), 2),
        "assists": p["assists"],
        "xA": round(float(p["xA"]), 2),
        "npxG": round(float(p["npxG"]), 2),
        "shots": p["shots"],
        "key_passes": p["key_passes"],
    }


def get_player_xg(player_name: str, comp_level: str, season: str, team_hint: str = "") -> dict | None:
    """High-level lookup: FBref-style comp_level/season -> Understat xG data, or None if the
    league isn't covered by Understat or the player isn't found in it."""
    league = fbref_comp_to_understat_league(comp_level)
    if league is None:
        return None
    year = fbref_season_to_understat_year(season)
    players = fetch_league_players(league, year)
    return find_player_xg(players, player_name, team_hint)


In [ ]:
%%writefile news_fetch.py
#!/usr/bin/env python3
"""
ScoutLite: player/team name -> recent news headlines. Uses NewsAPI's free "Developer" tier:
100 requests/day, dev/test use only (not licensed for production or internal production use),
1-month article lookback. robots.txt disallows /v1/ and /v2/ from crawlers, but that's
standard index-avoidance for an API-first service, not a restriction on calling the API with
a registered key.
"""
import argparse
import os
import sys
from datetime import datetime, timedelta
from pathlib import Path

import requests
from dotenv import load_dotenv
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

ROOT = Path(__file__).resolve().parent
load_dotenv(ROOT / ".env")

API_URL = "https://newsapi.org/v2/everything"
LOOKBACK_DAYS = 28  # stay safely inside the Developer tier's 1-month limit
ARTICLE_LIMIT = 15


def fetch_articles(query: str, api_key: str, limit: int = ARTICLE_LIMIT) -> list[dict]:
    from_date = (datetime.utcnow() - timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d")
    response = requests.get(
        API_URL,
        params={
            "q": query,
            "from": from_date,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": limit,
            "apiKey": api_key,
        },
        timeout=15,
    )
    response.raise_for_status()
    return response.json().get("articles", [])


def main():
    parser = argparse.ArgumentParser(
        description="Pull recent news headlines for a player/team and score basic sentiment"
    )
    parser.add_argument("query", help="Player or team name, e.g. 'Erling Haaland'")
    args = parser.parse_args()

    api_key = os.environ.get("NEWSAPI_KEY")
    if not api_key:
        sys.exit("NEWSAPI_KEY is not set. Add it to .env in this project folder.")

    print(f"Fetching news for '{args.query}' (last {LOOKBACK_DAYS} days)...")
    articles = fetch_articles(args.query, api_key)

    if not articles:
        print("No articles found.")
        return

    analyzer = SentimentIntensityAnalyzer()
    print(f"\nFound {len(articles)} articles:\n")
    scores = []
    for article in articles:
        text = f"{article.get('title', '')} {article.get('description', '') or ''}"
        score = analyzer.polarity_scores(text)["compound"]
        scores.append(score)
        source = article.get("source", {}).get("name", "unknown")
        published = article.get("publishedAt", "")[:10]
        print(f"[{score:+.2f}] {article.get('title', '')}  ({source}, {published})")

    avg = sum(scores) / len(scores)
    print(f"\nAverage sentiment: {avg:+.2f} (-1 very negative, +1 very positive)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scoring.py
#!/usr/bin/env python3
"""
ScoutLite Quality signal (1-5): a non-AI, rule-based baseline per the Technical Vision doc --
percentile-rank the player's per-90 stats against a real reference population (same league +
season), average the percentiles for their position group, map to 1-5 via fixed cutoffs.
No LLM involved in this number at all; the LLM is reserved for the Fit signal and prose.

Position groups (revised from the original doc spec after verifying data availability --
progressive passes/carries and a true duel-success-rate are NOT available on free FBref pages,
confirmed against both FBref's own "On this page" table-of-contents and Understat's actual
getPlayerData API response, not just their page labels -- see NOTES.md):
  - Attack:     goals/90, assists/90, xG/90, xA/90                    (Understat population)
  - Midfield:   key_passes/90, xA/90, (interceptions+tackles_won)/90  (Understat + FBref misc)
  - Defense:    (interceptions+tackles_won)/90                       (FBref misc population)
  - Goalkeeper: save% (already a rate, no per-90 needed)              (FBref keeper population)

Each metric's percentile is computed against its own single source's full population --
deliberately avoids merging Understat and FBref player-by-player for the reference population
(that name-matching fragility is only worth accepting for OUR ONE target player, not for
every player in a 400+ player league population where a few mismatches would be invisible
noise anyway).
"""
import re

import soccerdata as sd

from understat_xg import fbref_comp_to_understat_league, fbref_season_to_understat_year, fetch_league_players

SOCCERDATA_LEAGUES = {
    "premier league": "ENG-Premier League",
    "la liga": "ESP-La Liga",
    "bundesliga": "GER-Bundesliga",
    "serie a": "ITA-Serie A",
    "ligue 1": "FRA-Ligue 1",
}

MIN_MINUTES_FOR_POPULATION = 450  # ~5 full matches -- excludes small-sample noise from the reference


def fbref_comp_to_soccerdata_league(comp_level: str) -> str | None:
    name = comp_level.split(".", 1)[-1].strip().lower()
    return SOCCERDATA_LEAGUES.get(name)


def classify_position_group(position: str) -> str | None:
    """FBref position strings look like 'FW-MF', 'DF (CB)', 'GK', 'MF'. Takes the first
    (primary) position code FBref lists."""
    if not position:
        return None
    code = re.split(r"[-\s(]", position.strip())[0].upper()
    return {"GK": "goalkeeper", "DF": "defense", "MF": "midfield", "FW": "attack"}.get(code)


def per90(count: float, minutes: float) -> float:
    if not minutes:
        return 0.0
    return count / (minutes / 90)


def percentile_rank(value: float, population: list[float]) -> float:
    """% of the population this value is greater than or equal to. 0 population -> 50 (neutral)."""
    population = [p for p in population if p is not None]
    if not population:
        return 50.0
    below_or_equal = sum(1 for p in population if p <= value)
    return 100 * below_or_equal / len(population)


def percentile_to_1_5(pct: float) -> int:
    if pct < 20:
        return 1
    if pct < 40:
        return 2
    if pct < 60:
        return 3
    if pct < 80:
        return 4
    return 5


def _understat_population_per90(players: list[dict], position_group: str, field: str) -> list[float]:
    """Understat's 'position' field is a space-separated set of every position a player
    appeared in this season (e.g. 'F M S'), not a single primary tag -- match by containment,
    not equality, or most players get excluded entirely."""
    group_code = {"attack": "F", "midfield": "M", "defense": "D", "goalkeeper": "GK"}[position_group]
    return [
        per90(float(p[field]), float(p["time"]))
        for p in players
        if group_code in p.get("position", "").split() and float(p["time"]) >= MIN_MINUTES_FOR_POPULATION
    ]


def _fbref_misc_population_per90(misc_df, position_group: str) -> list[float]:
    pos_prefix = {"midfield": "MF", "defense": "DF"}[position_group]
    rows = misc_df[misc_df[("pos", "")].astype(str).str.startswith(pos_prefix)]
    out = []
    min_nineties = MIN_MINUTES_FOR_POPULATION / 90
    for _, row in rows.iterrows():
        nineties = row[("90s", "")]
        if not nineties or nineties < min_nineties:
            continue
        interceptions = row[("Performance", "Int")] or 0
        tackles_won = row[("Performance", "TklW")] or 0
        out.append((float(interceptions) + float(tackles_won)) / float(nineties))
    return out


def _fbref_keeper_population(keeper_df) -> list[float]:
    min_nineties = MIN_MINUTES_FOR_POPULATION / 90
    rows = keeper_df[keeper_df[("Playing Time", "90s")] >= min_nineties]
    return [float(v) for v in rows[("Performance", "Save%")].dropna().tolist()]


def compute_quality_signal(
    position: str,
    comp_level: str,
    season: str,
    player_name: str,
    stats: dict,
    misc: dict | None,
    keeper: dict | None,
    xg: dict | None,
) -> dict | None:
    """Returns {'score': 1-5, 'position_group': str, 'components': {metric: percentile}} or
    None if the league isn't covered or the position/data needed isn't available."""
    group = classify_position_group(position)
    if group is None:
        return None

    understat_league = fbref_comp_to_understat_league(comp_level)
    soccerdata_league = fbref_comp_to_soccerdata_league(comp_level)
    components = {}

    if group == "goalkeeper":
        if soccerdata_league is None or keeper is None or not keeper.get("save_pct"):
            return None
        sd_reader = sd.FBref(leagues=soccerdata_league, seasons=season)
        keeper_pop = _fbref_keeper_population(sd_reader.read_player_season_stats(stat_type="keeper"))
        components["save_pct"] = percentile_rank(float(keeper["save_pct"]), keeper_pop)

    else:
        minutes = float(str(stats.get("minutes", "0")).replace(",", "") or 0)
        if minutes == 0:
            return None

        if group in ("attack", "midfield") and understat_league and xg:
            year = fbref_season_to_understat_year(season)
            players = fetch_league_players(understat_league, year)
            if group == "attack":
                for field, key in [("goals", "goals_per90"), ("assists", "assists_per90"), ("xG", "xG_per90"), ("xA", "xA_per90")]:
                    pop = _understat_population_per90(players, group, field)
                    components[key] = percentile_rank(per90(float(xg[field]), float(xg["minutes"])), pop)
            elif group == "midfield":
                kp_pop = _understat_population_per90(players, group, "key_passes")
                xa_pop = _understat_population_per90(players, group, "xA")
                components["key_passes_per90"] = percentile_rank(per90(float(xg["key_passes"]), float(xg["minutes"])), kp_pop)
                components["xA_per90"] = percentile_rank(per90(float(xg["xA"]), float(xg["minutes"])), xa_pop)

        if group in ("midfield", "defense") and soccerdata_league and misc:
            sd_reader = sd.FBref(leagues=soccerdata_league, seasons=season)
            misc_df = sd_reader.read_player_season_stats(stat_type="misc")
            def_pop = _fbref_misc_population_per90(misc_df, group)
            def_value = per90(
                float(misc.get("interceptions", 0) or 0) + float(misc.get("tackles_won", 0) or 0), minutes
            )
            components["defensive_actions_per90"] = percentile_rank(def_value, def_pop)

    if not components:
        return None

    avg_pct = sum(components.values()) / len(components)
    return {"score": percentile_to_1_5(avg_pct), "position_group": group, "components": components, "avg_percentile": round(avg_pct, 1)}


In [ ]:
%%writefile docx_report.py
#!/usr/bin/env python3
"""
Builds the ScoutLite player research brief as a .docx file.

Structure follows the Technical Vision doc:
  - Signals block at the top (Quality X/5 + Fit X/5 = X/10, always with breakdown -- never
    shown alone). Quality is a non-AI percentile-based baseline (scoring.py); Fit is the LLM's
    numeric read. Either can come back None (unsupported league/position, or no philosophy
    given) -- shown as "not available" rather than a fabricated number.
  - 1. Who he is -- bio/background (pure data, no LLM)
  - 2. Stats & performance -- season/misc/keeper/xG tables (pure data, no LLM)
  - 3. What people say -- news headlines (data) + a short LLM-synthesized read
  - 4. Signals & fit read -- LLM fit-signal paragraph + scout's role notes + philosophy chosen

Deliberately named "research brief," never "report" or "verdict" -- ScoutLite is a data/
research layer for a scout, not a conclusion reached on their behalf.
"""
import re
from pathlib import Path

from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt


def _add_dict_table(doc: Document, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for key, value in data.items():
        if value in (None, ""):
            continue
        row = table.add_row().cells
        row[0].text = key.replace("_", " ").title()
        row[1].text = str(value)


def build_docx(
    player_name: str,
    bio: dict,
    stats: dict,
    xg: dict | None,
    articles: list[dict],
    misc: dict | None,
    keeper: dict | None,
    news_synthesis: str,
    fit_read: str,
    scout_notes: str | None,
    philosophy: dict | None,
    output_path: Path,
    quality: dict | None = None,
    fit_score: int | None = None,
) -> Path:
    doc = Document()

    title = doc.add_heading(f"{player_name} — Player Research Brief", level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.LEFT

    subtitle = doc.add_paragraph()
    subtitle.add_run(
        f"Season: {stats.get('season', 'unknown')}  ·  A data/research layer for a scout to "
        "weigh -- not a scouting verdict."
    ).italic = True

    # --- Signals block (top) -----------------------------------------------------------
    doc.add_heading("Signals", level=1)
    quality_score = quality["score"] if quality else None
    combined = f"{quality_score + fit_score}/10" if quality_score and fit_score else "—/10"
    p = doc.add_paragraph()
    p.add_run(
        f"Quality signal: {quality_score if quality_score else 'not available'}/5  ·  "
        f"Fit signal: {fit_score if fit_score else 'not available'}/5  ·  Combined: {combined}"
    ).bold = True
    doc.add_paragraph().add_run(
        "Signals for the scout to weigh, never a conclusion the tool reaches on the scout's "
        "behalf. Quality is a non-AI, percentile-based baseline (this player's per-90 stats vs. "
        "the same league/season's real players in their position group) -- not an LLM judgment. "
        "Fit is the LLM's read of stats + role notes against the club philosophy, when given."
    ).italic = True
    if quality:
        doc.add_paragraph().add_run(
            f"Quality breakdown ({quality['position_group']} group, "
            f"{quality['avg_percentile']} percentile average): " + ", ".join(
                f"{k.replace('_', ' ')} = {v:.0f} percentile" for k, v in quality["components"].items()
            )
        ).italic = True
    elif quality_score is None:
        doc.add_paragraph().add_run(
            "Quality signal not available -- either this player's league isn't one of the 5 "
            "covered (Premier League, La Liga, Bundesliga, Serie A, Ligue 1), or there wasn't "
            "enough data for their position group this season."
        ).italic = True

    # --- 1. Who he is --------------------------------------------------------------------
    doc.add_heading("1. Who He Is", level=1)
    if bio:
        _add_dict_table(doc, bio)
    else:
        doc.add_paragraph("No background data available.")

    # --- 2. Stats & performance ------------------------------------------------------------
    doc.add_heading("2. Stats & Performance", level=1)
    doc.add_heading(f"Season stats ({stats.get('season', 'unknown')})", level=2)
    _add_dict_table(doc, stats)

    if keeper:
        doc.add_heading("Goalkeeping stats", level=2)
        _add_dict_table(doc, keeper)

    if misc:
        doc.add_heading("Defensive/discipline stats", level=2)
        _add_dict_table(doc, misc)

    if xg:
        doc.add_heading("Advanced stats (Understat: xG/xA)", level=2)
        _add_dict_table(doc, xg)
    else:
        doc.add_paragraph("Advanced stats (xG/xA): not available -- Understat doesn't cover this player's league.")

    # --- 3. What people say --------------------------------------------------------------
    doc.add_heading("3. What People Say", level=1)
    doc.add_paragraph(news_synthesis or "No recent news synthesis available.")
    if articles:
        doc.add_heading("Recent headlines (last 28 days)", level=2)
        for a in articles[:8]:
            doc.add_paragraph(a.get("title", ""), style="List Bullet")

    # --- 4. Signals & fit read -------------------------------------------------------------
    doc.add_heading("4. Signals & Fit Read", level=1)
    if philosophy and (philosophy.get("in_possession") or philosophy.get("out_of_possession")):
        style_desc = " / ".join(v for v in philosophy.values() if v)
        doc.add_paragraph().add_run(
            f"Club philosophy assessed against: {style_desc}  ·  Fit signal: {fit_score if fit_score else 'not available'}/5"
        ).bold = True
    if scout_notes and scout_notes.strip():
        doc.add_paragraph().add_run(f"Scout's role notes: \"{scout_notes.strip()}\"").italic = True
    doc.add_paragraph(fit_read or "No fit signal available.")

    doc.save(output_path)
    return output_path


In [ ]:
%%writefile scoutlite_combined.py
#!/usr/bin/env python3
"""
ScoutLite combined pipeline: one player name -> FBref (stats + bio) + Understat (xG/xA, when
the league is covered) + NewsAPI (recent headlines, last 28 days) -> one DeepSeek-V3 call ->
one player research brief (a data/research layer for a scout, not a scouting verdict).

Reuses each source's already-proven functions rather than duplicating logic:
- scoutlite.py: fetch_player_page_html, extract_latest_season, extract_player_bio
- understat_xg.py: get_player_xg (returns None if the league isn't one Understat tracks)
- news_fetch.py: fetch_articles (returns [] if none found; genuinely recent only, not
  season-long -- NewsAPI's free tier caps lookback at ~1 month)

Understat access note: understat.com's robots.txt blanket-disallows all scraping. Fetching it
here is a deliberate, explicit override made by the project owner -- see NOTES.md.
"""
import argparse
import os
import re
import sys
from pathlib import Path

import requests
from dotenv import load_dotenv
from openai import OpenAI

from docx_report import build_docx
from news_fetch import LOOKBACK_DAYS, fetch_articles
from scoutlite import (
    extract_keeper_stats,
    extract_latest_season,
    extract_misc_stats,
    extract_player_bio,
    fetch_player_page_html,
)
from scoring import compute_quality_signal
from understat_xg import get_player_xg

ROOT = Path(__file__).resolve().parent
load_dotenv(ROOT / ".env")

ROLE_NOTES_MAX_CHARS = 200  # short, adjective-style role notes -- not a full report


NEWS_MARKER = "###WHAT_PEOPLE_SAY###"
FIT_MARKER = "###SIGNALS_AND_FIT_READ###"


def build_prompt(
    player_name: str,
    stats: dict,
    xg: dict | None,
    articles: list[dict],
    misc: dict | None = None,
    keeper: dict | None = None,
    scout_notes: str | None = None,
    philosophy: dict | None = None,
) -> str:
    """Bio and stats are rendered as tables directly from data elsewhere -- no LLM restatement,
    no transcription risk. This prompt only asks for the two things that genuinely need
    synthesis: a read on recent news, and a fit signal against the club's philosophy."""
    stats_lines = "\n".join(f"- {k.replace('_', ' ')}: {v}" for k, v in stats.items())
    sections = [f"Player: {player_name}", f"\nSeason stats ({stats.get('season', 'unknown season')}):\n{stats_lines}"]

    if keeper:
        keeper_lines = "\n".join(f"- {k.replace('_', ' ')}: {v}" for k, v in keeper.items())
        sections.append(f"\nGoalkeeping stats (same season):\n{keeper_lines}")

    if misc:
        misc_lines = "\n".join(f"- {k.replace('_', ' ')}: {v}" for k, v in misc.items())
        sections.append(f"\nDefensive/discipline stats (same season):\n{misc_lines}")

    if xg:
        xg_lines = "\n".join(f"- {k.replace('_', ' ')}: {v}" for k, v in xg.items())
        sections.append(f"\nAdvanced stats (expected goals/assists, same season):\n{xg_lines}")
    else:
        sections.append(
            "\nAdvanced stats (xG/xA): not available -- Understat doesn't cover this player's league."
        )

    if articles:
        headlines = "\n".join(f"- {a.get('title', '')}" for a in articles[:8])
        sections.append(
            f"\nRecent news headlines (last {LOOKBACK_DAYS} days only -- NOT a full season of "
            f"coverage, do not treat as season-long context):\n{headlines}"
        )
    else:
        sections.append(f"\nRecent news (last {LOOKBACK_DAYS} days): none found.")

    if scout_notes and scout_notes.strip():
        notes = scout_notes.strip()[:ROLE_NOTES_MAX_CHARS]
        sections.append(
            "\nScout's own short role notes (subjective, capped free text from a human scout who "
            "has watched this player -- captures ROLE, e.g. how they're actually used on the pitch, "
            "which stats alone don't show; not verified data, do not fact-check or contradict it, "
            f"just incorporate it as the scout's observation):\n{notes}"
        )

    has_philosophy = philosophy and (philosophy.get("in_possession") or philosophy.get("out_of_possession"))
    if has_philosophy:
        style_desc = " / ".join(v for v in philosophy.values() if v)
        sections.append(f"\nClub philosophy to assess fit against: {style_desc}")

    instructions = [
        "You are ScoutLite, generating two short sections of a player research brief -- a "
        "data/research layer FOR a scout, not a scouting verdict. Using ONLY the information "
        "listed below, produce exactly two labeled sections, in this format:\n\n"
        f"{NEWS_MARKER}\n"
        "One short paragraph reading the recent news (last 28 days). If none is relevant, say so "
        "plainly rather than inventing significance. Do not treat this as season-long context.\n\n"
        f"{FIT_MARKER}\n"
    ]

    if has_philosophy:
        instructions.append(
            f"First line MUST be exactly \"Fit: X/5\" where X is your integer 1-5 fit signal for a "
            f"club playing {style_desc}, based ONLY on the stats already listed above (1 = poor fit, "
            f"3 = neutral/insufficient data to tell, 5 = excellent fit). Then, on a new paragraph, "
            f"explain that score using ONLY those stats (e.g. defensive actions relate to pressing "
            f"demands, key passes/crosses relate to possession play). Explicitly say when the "
            f"available stats aren't sufficient to judge a given aspect (e.g. no pace/sprint data "
            f"here, so speed-dependent transition fit can't be assessed) rather than guessing -- "
            f"when in doubt, score closer to 3, not a confident extreme. If the scout's role notes "
            f"are present, weave them into the explanation as the scout's own observation, clearly "
            f"attributed, not verified fact. This is a fit SIGNAL for the scout to weigh, never a "
            f"verdict."
        )
    else:
        instructions.append(
            "First line MUST be exactly \"Fit: not assessed\" (no club philosophy was specified). "
            "Then, on a new paragraph, note the scout's role notes if present, clearly attributed "
            "as the scout's own observation, not verified fact."
        )

    instructions.append(
        "\n\nDo not invent or infer any fact, stat, or event not listed below. Do not speculate "
        "about transfer value, potential, or future performance. Never present anything as a "
        "conclusion or verdict -- these are signals for the scout to weigh."
    )

    body = "\n".join(s for s in sections if s)
    return "".join(instructions) + "\n\n" + body


def summarize_combined(
    player_name: str,
    stats: dict,
    xg: dict | None,
    articles: list[dict],
    misc: dict | None = None,
    keeper: dict | None = None,
    scout_notes: str | None = None,
    philosophy: dict | None = None,
) -> dict:
    """Returns {'news_synthesis': str, 'fit_read': str, 'fit_score': int|None} -- the two
    sections that need LLM synthesis. Bio/stats/news-list are rendered as tables directly from
    data, not through here."""
    client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"], base_url="https://api.deepseek.com")
    prompt = build_prompt(player_name, stats, xg, articles, misc, keeper, scout_notes, philosophy)
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
    )
    text = response.choices[0].message.content.strip()

    news_synthesis, fit_read = "", ""
    if NEWS_MARKER in text and FIT_MARKER in text:
        news_synthesis = text.split(NEWS_MARKER, 1)[1].split(FIT_MARKER, 1)[0].strip()
        fit_read = text.split(FIT_MARKER, 1)[1].strip()
    else:
        # Model didn't follow the marker format -- surface the raw text rather than lose it.
        news_synthesis = text

    fit_score = None
    fit_score_match = re.match(r"Fit:\s*(\d)/5", fit_read)
    if fit_score_match:
        fit_score = int(fit_score_match.group(1))
        fit_read = fit_read[fit_score_match.end():].strip()
    elif fit_read.lower().startswith("fit: not assessed"):
        fit_read = fit_read[len("Fit: not assessed"):].strip()

    return {"news_synthesis": news_synthesis, "fit_read": fit_read, "fit_score": fit_score}


def main():
    parser = argparse.ArgumentParser(
        description="Combined ScoutLite pipeline: FBref + Understat + NewsAPI -> one summary"
    )
    parser.add_argument("player", help="Player name, e.g. 'Erling Haaland'")
    parser.add_argument("--season", help="Specific season to report on, e.g. '2024-2025' (default: most recent)")
    parser.add_argument("--scout-notes", help="Your own short role notes (capped, adjective-style)")
    parser.add_argument("--in-possession", choices=["vertical", "possession"], help="Club philosophy: in-possession axis")
    parser.add_argument("--out-of-possession", choices=["high_line", "low_block", "mid_block"], help="Club philosophy: out-of-possession axis")
    args = parser.parse_args()

    if not os.environ.get("DEEPSEEK_API_KEY"):
        sys.exit("DEEPSEEK_API_KEY is not set. Add it to .env in this project folder.")

    philosophy = {
        "in_possession": {
            "vertical": "vertical, fast transitions",
            "possession": "slow, methodical possession",
        }.get(args.in_possession, ""),
        "out_of_possession": {
            "high_line": "high line, counter-press",
            "low_block": "low block, counter",
            "mid_block": "mid block, hybrid",
        }.get(args.out_of_possession, ""),
    }

    print(f"Fetching FBref data for '{args.player}'...")
    url, html = fetch_player_page_html(args.player)
    print(f"Resolved to: {url}")

    stats = extract_latest_season(html, args.season)
    bio = extract_player_bio(html)
    misc = extract_misc_stats(html, args.season)
    keeper = extract_keeper_stats(html, args.season)
    print(f"Season: {stats['season']} ({stats['squad']}, {stats['competition']})")
    print(f"Goalkeeping stats: {'found' if keeper else 'not applicable'}")

    print("Looking up Understat xG/xA...")
    xg = get_player_xg(args.player, stats["competition"], stats["season"], stats["squad"])
    print(f"xG/xA: {'found' if xg else 'not available for this league'}")

    print("Computing Quality signal (non-AI, percentile-based)...")
    quality = compute_quality_signal(
        bio["position"], stats["competition"], stats["season"], args.player, stats, misc, keeper, xg
    )
    print(f"Quality: {quality['score'] if quality else 'not available for this league/position'}")

    articles = []
    newsapi_key = os.environ.get("NEWSAPI_KEY")
    if newsapi_key:
        print(f"Fetching recent news (last {LOOKBACK_DAYS} days)...")
        try:
            articles = fetch_articles(args.player, newsapi_key)
            print(f"Found {len(articles)} articles")
        except requests.RequestException as e:
            print(f"NewsAPI request failed ({e}) -- continuing without news")
    else:
        print("NEWSAPI_KEY not set -- skipping news, continuing without it")

    print("Calling DeepSeek-V3 for the research brief...")
    synthesis = summarize_combined(
        args.player, stats, xg, articles, misc, keeper, args.scout_notes, philosophy
    )

    output_dir = ROOT / "output"
    output_dir.mkdir(exist_ok=True)
    slug = re.sub(r"[^a-z0-9]+", "_", args.player.lower()).strip("_")
    out_path = output_dir / f"{slug}_brief.docx"
    build_docx(
        args.player, bio, stats, xg, articles, misc, keeper,
        synthesis["news_synthesis"], synthesis["fit_read"],
        args.scout_notes, philosophy, out_path,
        quality=quality, fit_score=synthesis["fit_score"],
    )

    print(f"\nQuality: {quality['score'] if quality else 'N/A'}/5  ·  Fit: {synthesis['fit_score'] or 'N/A'}/5")
    print("\n--- What People Say ---")
    print(synthesis["news_synthesis"])
    print("\n--- Signals & Fit Read ---")
    print(synthesis["fit_read"])
    print(f"\nSaved to {out_path}")


if __name__ == "__main__":
    main()


## 3. API keys

Two required (`DEEPSEEK_API_KEY`, and optionally `NEWSAPI_KEY` for the news section — the pipeline runs fine without it, just skips that section).

**Recommended:** use Colab's built-in Secrets manager (key icon in the left sidebar) to add `DEEPSEEK_API_KEY` and `NEWSAPI_KEY`, then toggle "Notebook access" on for each. The cell below reads them from there if present, otherwise falls back to a hidden prompt so nothing is ever typed into a visible cell or saved into the notebook file itself.

In [ ]:
import os
from getpass import getpass

def _get_secret(name: str, required: bool = True) -> str | None:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    value = os.environ.get(name)
    if value:
        return value
    if required:
        return getpass(f"{name} (not found in Colab secrets, paste it here): ")
    return None

os.environ["DEEPSEEK_API_KEY"] = _get_secret("DEEPSEEK_API_KEY")
newsapi_key = _get_secret("NEWSAPI_KEY", required=False)
if newsapi_key:
    os.environ["NEWSAPI_KEY"] = newsapi_key
    print("NEWSAPI_KEY set -- news section will run.")
else:
    print("NEWSAPI_KEY not set -- pipeline will skip the news section (this is fine, it's optional).")

## 4. Run the pipeline

Set `PLAYER_NAME` (and optionally `SEASON`, scout notes, and a club philosophy) below, then run. This calls the exact same functions the local Streamlit app and CLI use — nothing notebook-specific about the pipeline logic itself.

The FBref step is paced (~7-9s, by design — see the intro) and is the one step whose Colab behavior is unverified (the Cloudflare/cloud-IP caveat above).

In [ ]:
PLAYER_NAME = "Erling Haaland"
SEASON = None  # e.g. "2023-2024" -- None uses the most recent season
SCOUT_NOTES = ""  # e.g. "tall, can play as a 9 or 10, fast feet, gets in behind often"
IN_POSSESSION = None  # "vertical" | "possession" | None
OUT_OF_POSSESSION = None  # "high_line" | "low_block" | "mid_block" | None

philosophy = {
    "in_possession": {
        "vertical": "vertical, fast transitions",
        "possession": "slow, methodical possession",
    }.get(IN_POSSESSION, ""),
    "out_of_possession": {
        "high_line": "high line, counter-press",
        "low_block": "low block, counter",
        "mid_block": "mid block, hybrid",
    }.get(OUT_OF_POSSESSION, ""),
}

print(f"Fetching FBref data for '{PLAYER_NAME}'... (paced, ~7-9s -- this is the Cloudflare-dependent step)")
from scoutlite import fetch_player_page_html, extract_latest_season, extract_player_bio, extract_misc_stats, extract_keeper_stats

url, html = fetch_player_page_html(PLAYER_NAME)
print(f"Resolved to: {url}")

stats = extract_latest_season(html, SEASON)
bio = extract_player_bio(html)
misc = extract_misc_stats(html, SEASON)
keeper = extract_keeper_stats(html, SEASON)
print(f"Season: {stats['season']} ({stats['squad']}, {stats['competition']})")

In [ ]:
from understat_xg import get_player_xg
from scoring import compute_quality_signal

print("Looking up Understat xG/xA...")
xg = get_player_xg(PLAYER_NAME, stats["competition"], stats["season"], stats["squad"])
print("found" if xg else "not available for this league")

print("\nComputing Quality signal (non-AI, percentile-based)...")
quality = compute_quality_signal(
    bio["position"], stats["competition"], stats["season"], PLAYER_NAME, stats, misc, keeper, xg
)
print(f"Quality: {quality['score']}/5" if quality else "Quality signal not available (league/position not covered)")

In [ ]:
from news_fetch import fetch_articles, LOOKBACK_DAYS

articles = []
if os.environ.get("NEWSAPI_KEY"):
    print(f"Fetching recent news (last {LOOKBACK_DAYS} days)...")
    try:
        articles = fetch_articles(PLAYER_NAME, os.environ["NEWSAPI_KEY"])
        print(f"Found {len(articles)} articles")
    except Exception as e:
        print(f"NewsAPI request failed ({e}) -- continuing without news")
else:
    print("NEWSAPI_KEY not set -- skipping news")

In [ ]:
from scoutlite_combined import summarize_combined

print("Calling DeepSeek-V3 for the research brief (news read + Fit signal)...")
synthesis = summarize_combined(PLAYER_NAME, stats, xg, articles, misc, keeper, SCOUT_NOTES, philosophy)

print(f"\nQuality: {quality['score'] if quality else 'N/A'}/5  ·  Fit: {synthesis['fit_score'] or 'N/A'}/5\n")
print("--- What People Say ---")
print(synthesis["news_synthesis"])
print("\n--- Signals & Fit Read ---")
print(synthesis["fit_read"])

## 5. Build and download the Word doc research brief

In [ ]:
import re
from pathlib import Path
from docx_report import build_docx

slug = re.sub(r"[^a-z0-9]+", "_", PLAYER_NAME.lower()).strip("_")
out_path = Path(f"{slug}_brief.docx")
build_docx(
    PLAYER_NAME, bio, stats, xg, articles, misc, keeper,
    synthesis["news_synthesis"], synthesis["fit_read"],
    SCOUT_NOTES, philosophy, out_path,
    quality=quality, fit_score=synthesis["fit_score"],
)
print(f"Saved to {out_path}")

try:
    from google.colab import files
    files.download(str(out_path))
except Exception:
    print("(Not running in Colab, or download was blocked -- the file is saved locally at the path above.)")

---

### Data source access, in brief

| Source | Status | Why |
|---|---|---|
| FBref | ✅ Core source | Permitted, rate-limited (paced ~7-9s here) |
| Understat | ⚠️ Deliberate override | `robots.txt` blanket-disallows scraping; used anyway, explicit project-owner decision, disclosed throughout |
| NewsAPI | ✅ Official API | Free tier, optional |
| Transfermarkt | ❌ Not used | Named block on `ClaudeBot`/`anthropic-ai` in `robots.txt` |
| FotMob | ❌ Not used | `robots.txt` disallow on data endpoints |
| Reddit, X | ❌ Not used | Blanket `robots.txt` disallow (Reddit); same + paid-only API (X) |

Full reasoning for every decision above lives in the project's `NOTES.md`. This notebook is a demo of the pipeline, not the canonical source — the local project (with the Streamlit UI, `.env`-based keys, and the isolated venv launcher) is that.